# 15-nt UTR parent screen: non-A at the A15 junction

This revision applies your new condition to the **A/AU sample pool**: each selected parent has exactly the final 15 bases of its extracted 50-nt UTR window followed by A15. The final UTR base must be U, C or G, so Muller position −1 is immediately before A15. The five A/AU motifs and the A/U-only −15 through −2 rule are unchanged.

The three single-position categories are −13 to −9, −11 to −7, and −6 to −2; a separate both-ends category requires motifs at −15 to −11 and −6 to −2. A position group enters the top-two panel only if all five motifs are represented at that position. Incomplete groups are reported separately and not selected as complete groups. The synthetic A30 control is retained in its own sheet and has no non-A position −1.

The separate benchmark workbook keeps the previous benchmark rules so existing benchmarks are not silently changed. A 15-nt model segment does not establish that the biological transcript's complete 3′UTR is 15 nt. Extracted hg19 windows are contiguous genomic sequence, not splice-aware. This reconstruction is not the authors' exact 38,700-sequence library.

In Colab run the cells in order. The notebook obtains polyA_DB v3.2 and UCSC hg19 and verifies their checksums. It downloads a ZIP with the sample and benchmark workbooks when complete.


## 1. Install required packages

In [ ]:
import sys, subprocess
try:
    import py2bit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py2bit==1.0.1'], check=True)
    import py2bit
try:
    import openpyxl
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], check=True)
    import openpyxl
print('Packages ready.')


## 2. Define the database extraction and ranking rules

In [ ]:
import csv, hashlib
from decimal import Decimal, InvalidOperation
from collections import Counter
from itertools import product
from pathlib import Path
from datetime import datetime

CG_MOTIFS = ('GGCCG', 'CCGCG', 'GCGCG', 'CGGCG', 'CGCGG')

AU_MOTIFS = ('AAAAA', 'AAAAU', 'UAAAA', 'UAUAA', 'AUAUA')

CONTROL_ID = 'chr5:134907433:-'

CONTROL_SEQUENCE = 'A' * 30

PAS_SHA256 = 'c20167a841763b2977376467ef80014765ef595db93f2a023fb2d05a9c39b73d'

HG19_MD5 = 'bcdbfbe9da62f19bee88b74dabef8cd3'

HG19_BYTES = 816241703

COMPLEMENT = str.maketrans('ACGT', 'TGCA')

def digest(path, name):
    h = hashlib.new(name)
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract50(genome, chrom, pos, strand):
    chroms = genome.chroms()
    if chrom not in chroms:
        raise ValueError('chromosome_missing')
    if strand == '+':
        start, end = pos - 50, pos
    elif strand == '-':
        start, end = pos - 1, pos + 49
    else:
        raise ValueError('invalid_strand')
    if start < 0 or end > chroms[chrom]:
        raise ValueError('chromosome_boundary')
    seq = genome.sequence(chrom, start, end)
    if seq is None or len(seq) != 50:
        raise ValueError('reference_read_failed_or_short')
    seq = seq.upper()
    if set(seq) - set('ACGT'):
        raise ValueError('ambiguous_reference_bases')
    if strand == '-':
        seq = seq.translate(COMPLEMENT)[::-1]
    return seq.replace('T', 'U')

def select_panel(rows, motifs, target, excluded_sequence=None):
    """Four per group where available; fill shortages by global score.

    A single 30-nt sequence contributes at most once; the highest scoring
    source PAS is retained, then lexical PAS ID breaks ties.
    """
    candidates = sorted((dict(r) for r in rows if r['motif'] in motifs
                         and r['utr15_A15'] != excluded_sequence),
                        key=lambda r: (-r['score'], r['PAS_ID']))
    unique, seen = [], set()
    for r in candidates:
        if r['utr15_A15'] not in seen:
            seen.add(r['utr15_A15']); unique.append(r)
    cap = target // len(motifs)
    result, selected = [], set()
    for motif in motifs:
        group = [r for r in unique if r['motif'] == motif]
        for r in group[:cap]:
            result.append(r); selected.add(r['utr15_A15'])
    for r in unique:
        if len(result) >= target: break
        if r['utr15_A15'] not in selected:
            result.append(r); selected.add(r['utr15_A15'])
    return result[:target]

def read_sites(path):
    """Collect each annotated 3'UTR PAS once; retain conservative membership."""
    groups = {}
    n_rows = 0
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f, delimiter='\t')
        required = {'PAS_ID','Chromosome','Position','Strand','Mean RPM',
                    'Conservation','PAS type','Gene Symbol'}
        missing = required - set(reader.fieldnames or [])
        if missing: raise ValueError(f'Missing columns in PAS table: {sorted(missing)}')
        for r in reader:
            n_rows += 1
            if not r['PAS type'].strip().startswith("3'UTR("):
                continue
            pas, chrom, strand = r['PAS_ID'].strip(), r['Chromosome'].strip(), r['Strand'].strip()
            try:
                pos, rpm = int(r['Position']), Decimal(r['Mean RPM'])
            except (ValueError, InvalidOperation) as e:
                raise ValueError(f'Invalid coordinate or mean RPM: {pas}') from e
            if pas != f'{chrom}:{pos}:{strand}' or strand not in ('+','-') or pos < 1 or not rpm.is_finite() or rpm < 0:
                raise ValueError(f'Invalid PAS annotation: {pas}')
            key = (chrom,pos,strand,rpm)
            g = groups.setdefault(pas, {'key':key, 'genes':set(), 'types':set(), 'conserved':False})
            if g['key'] != key: raise ValueError(f'Conflicting annotations for {pas}')
            symbol = r['Gene Symbol'].strip()
            if symbol and symbol.lower() not in ('nan','na','none'): g['genes'].add(symbol)
            g['types'].add(r['PAS type'].strip())
            g['conserved'] |= r['Conservation'].strip().lower() == 'yes'
    sites=[]
    for pas,g in groups.items():
        chrom,pos,strand,rpm=g['key']
        sites.append({'PAS_ID':pas,'Chromosome':chrom,'Position':pos,'Strand':strand,
                      'Mean_RPM':float(rpm),'Genes':';'.join(sorted(g['genes'])),
                      'PAS_types':';'.join(sorted(g['types'])),
                      'Conservation': 'Yes' if g['conserved'] else 'No'})
    return sites,n_rows

def unique_candidates(rows, excluded_sequence=None):
    seen=set();out=[]
    for r in sorted(rows,key=lambda x:(-x['score'],x['PAS_ID'])):
        if r['utr15_A15']==excluded_sequence or r['utr15_A15'] in seen: continue
        seen.add(r['utr15_A15']);out.append(r)
    return out


## 3. Obtain and verify the source files

In [ ]:
import os, shutil, urllib.request, zipfile

pas_path = Path('human.PAS.txt')
genome_path = Path('hg19.2bit')

def download_to(url, path):
    with urllib.request.urlopen(url, timeout=120) as response, open(path, 'wb') as dest:
        shutil.copyfileobj(response, dest, length=1024*1024)

if not pas_path.exists():
    print('Downloading the official polyA_DB v3.2 PAS table...')
    archive = Path('human_pas.zip.part')
    partial = Path('human.PAS.txt.part')
    try:
        download_to('https://exon.apps.wistar.org/PolyA_DB/v3/download/3.2/human_pas.zip', archive)
        with zipfile.ZipFile(archive) as z, z.open('human.PAS.txt') as source, partial.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        if digest(partial, 'sha256') != PAS_SHA256:
            raise ValueError('Downloaded PAS table checksum does not match the expected v3.2 file.')
        os.replace(partial, pas_path)
    finally:
        archive.unlink(missing_ok=True)
        partial.unlink(missing_ok=True)
if digest(pas_path, 'sha256') != PAS_SHA256:
    raise ValueError('Existing human.PAS.txt has the wrong checksum. Rename or remove it and rerun.')

if not genome_path.exists():
    print('Downloading UCSC hg19.2bit (~816 MB). This may take several minutes...')
    partial = Path('hg19.2bit.part')
    try:
        download_to('https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.2bit', partial)
        if partial.stat().st_size != HG19_BYTES or digest(partial, 'md5') != HG19_MD5:
            raise ValueError('Downloaded hg19 file failed checksum verification.')
        os.replace(partial, genome_path)
    finally:
        partial.unlink(missing_ok=True)
if genome_path.stat().st_size != HG19_BYTES or digest(genome_path, 'md5') != HG19_MD5:
    raise ValueError('Existing hg19.2bit failed verification. Rename or remove it and rerun.')
print('Both input files verified.')


## 4. Define the updated parent screening and selection

In [ ]:
import hashlib

SAMPLE_STARTS = (-13, -11, -6)


def anchor_info(seq):
    if len(seq) != 50 or set(seq) - set('ACGU'):
        raise ValueError('Expected a 50-nt RNA sequence.')
    anchor = len(seq.rstrip('A'))
    return (anchor if anchor else None, 50 - anchor)


def region14(seq):
    anchor, _ = anchor_info(seq)
    if anchor is None or anchor < 15:
        return None
    return seq[anchor-15:anchor-1]


def motif_at(seq, start):
    anchor, terminal_as = anchor_info(seq)
    if anchor is None or not -15 <= start <= -6:
        return None
    i = anchor + start
    if i < 0 or i+5 > anchor-1:
        return None
    return {'motif':seq[i:i+5], 'norm_start':start, 'norm_end':start+4,
            'junction_start':i-50, 'junction_end':i-46,
            'retained_in_15':i>=35 and i+5<=50}


def annotate(site, seq):
    anchor, terminal_as=anchor_info(seq)
    last15=seq[-15:]
    return {**site,
        'RNA_ID':'RNA_N'+hashlib.sha256((last15+'A'*15).encode('ascii')).hexdigest()[:12],
        'UTR50':seq,'UTR50_A15':seq+'A'*15,'UTR15':last15,'utr15_A15':last15+'A'*15,
        'terminal_UTR_As':terminal_as,'normalized_minus1_base':seq[anchor-1] if anchor else '',
        'minus1_junction_position':anchor-51 if anchor else '',
        'normalized_window_minus15_minus2':region14(seq) or '',
        'upstream_A_count':seq[:35].count('A'),
        'upstream_A_pct':round(seq[:35].count('A')*100/35,2),
        'CG_count_50':seq.count('C')+seq.count('G'),
        'CG_pct_50':2*(seq.count('C')+seq.count('G')),
        'CG_count_15':last15.count('C')+last15.count('G'),
        'CG_pct_15':round((last15.count('C')+last15.count('G'))*100/15,2)}


def run_screen(sites, genome):
    hits={p:[] for p in SAMPLE_STARTS};cg=[];au=[];failures=[]
    for site in sites:
        try:
            seq=extract50(genome,site['Chromosome'],site['Position'],site['Strand'])
        except ValueError as e:
            failures.append({'PAS_ID':site['PAS_ID'],'reason':str(e)});continue
        region=region14(seq)
        if region is None:continue
        row=annotate(site,seq)
        if site['Conservation']=='Yes' and set(region)<=set('AU'):
            for start in set(SAMPLE_STARTS+(-9,)):
                h=motif_at(seq,start)
                if not h or not h['retained_in_15'] or h['motif'] not in AU_MOTIFS:continue
                r={**row,**h,'score':row['upstream_A_count']}
                if start in SAMPLE_STARTS and seq[-1] in 'CGU':hits[start].append(r)
                if start==-9:au.append(r)
        if (row['CG_count_50']>=25 and row['CG_count_15']>=8
                and not any(m in region for m in AU_MOTIFS)):
            h=motif_at(seq,-9)
            if h and h['retained_in_15'] and h['motif'] in CG_MOTIFS:
                cg.append({**row,**h,'score':row['CG_count_50']})
    return {'hits':hits,'cg':cg,'au':au,'failures':failures}




In [ ]:
def build_groups(hits):
    coverage=[];counts=[];top=[];candidates=[];raw_hits=[];complete=[];balanced=[]
    for start in SAMPLE_STARTS:
        by_motif={m:unique_candidates([r for r in hits[start] if r['motif']==m],CONTROL_SEQUENCE) for m in AU_MOTIFS}
        n={m:len(by_motif[m]) for m in AU_MOTIFS}
        all_five=all(n[m]>=1 for m in AU_MOTIFS)
        two_each=all(n[m]>=2 for m in AU_MOTIFS)
        if all_five:complete.append(start);raw_hits.extend(hits[start])
        if two_each:balanced.append(start)
        coverage.append({'norm_start':start,'norm_end':start+4,**n,
                         'all_five_motifs':'Yes' if all_five else 'No',
                         'two_per_motif':'Yes' if two_each else 'No'})
        for m in AU_MOTIFS:
            group=by_motif[m]; selected=group[:2] if all_five else []
            counts.append({'norm_start':start,'norm_end':start+4,'motif':m,
                           'eligible_unique_RNAs':len(group),'selected_RNAs':len(selected)})
            if all_five:
                candidates.extend({**r,'Panel':'Complete position group'} for r in group)
                top.extend({**r,'Panel':'Complete position top two','selection_order':i} for i,r in enumerate(selected,1))
    return {'coverage':coverage,'counts':counts,'top':top,'candidates':candidates,
            'raw_hits':raw_hits,'pool':unique_candidates(raw_hits,CONTROL_SEQUENCE),
            'complete':complete,'balanced':balanced}


def choose_benchmark(data):
    cg=select_panel(data['cg'],CG_MOTIFS,20,excluded_sequence=CONTROL_SEQUENCE)
    au=select_panel(data['au'],AU_MOTIFS,20,excluded_sequence=CONTROL_SEQUENCE)
    au_unique=unique_candidates(data['au'],CONTROL_SEQUENCE)
    used={r['utr15_A15'] for r in au}
    choices=[r for r in au_unique if r['motif']=='AAAAA' and r['norm_start']==-9
             and r['retained_in_15'] and r['terminal_UTR_As']==0
             and r['normalized_minus1_base']=='U' and r['utr15_A15'] not in used]
    if not choices:raise ValueError('No distinct replacement reference satisfies the stated criteria.')
    reference=choices[0]
    panel=([{**r,'Panel':'CG benchmark','selection_order':i} for i,r in enumerate(cg,1)]
           +[{**r,'Panel':'AU benchmark','selection_order':i} for i,r in enumerate(au,1)]
           +[{**reference,'Panel':'Separate reference (ARHGEF2)'}])
    if len({r['utr15_A15'] for r in panel})!=len(panel):raise ValueError('Duplicate benchmark RNA detected.')
    return {'cg':cg,'au':au,'reference':reference,'panel':panel,
            'cg_unique':unique_candidates(data['cg'],CONTROL_SEQUENCE),'au_unique':au_unique}


FIELDS=['RNA_ID','Panel','selection_order','motif','norm_start','norm_end',
        'junction_start','junction_end','Genes','PAS_ID','Conservation',
        'UTR50_A15','UTR50','utr15_A15','UTR15','terminal_UTR_As',
        'normalized_minus1_base','minus1_junction_position',
        'normalized_window_minus15_minus2','upstream_A_count','upstream_A_pct',
        'CG_count_50','CG_pct_50','CG_count_15','CG_pct_15',
        'Mean_RPM','Chromosome','Position','Strand','PAS_types']




In [ ]:
def make_files(sites, data, sample_path, benchmark_path):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    groups=build_groups(data['hits']);bench=choose_benchmark(data)
    def add(w,name,rows,fields):
        s=w.create_sheet(name);s.append(fields)
        for r in rows:s.append([r.get(f,'') for f in fields])
        return s
    def make_summary(title,extra):
        w=Workbook();s=w.active;s.title='Summary';s.append(['Item','Value'])
        notes=[('Workflow',title),*extra,('Sites searched',len(sites)),
            ('Reference extraction failures',len(data['failures'])),
            ('Muller positions','The rightmost non-A base of the 50-nt genomic UTR is -1; positions count upstream.'),
            ('Sequence retained','The final 15 genomic UTR bases plus an appended A15; a selected motif must remain fully in those 15 bases.'),
            ('Two coordinates','norm_start/end are Muller positions. junction_start/end count from the designated genomic UTR/A15 junction.'),
            ('AU-only rule','Muller positions -15 to -2 contain only A or U; -1 is not restricted.'),
            ('CG requirements','50-nt UTR has at least 25 C/G; final 15 genomic UTR bases have at least 8 C/G; no target A/AU motif in normalized -15..-2.'),
            ('Ranking','CG: 50-nt C/G count. A/AU: A count of the first 35 bases of the original 50-nt UTR; ties use PAS ID. Those 35 bases are not in the 30-nt RNA.'),
            ('Source scopes','CG: all annotated 3UTR sites; AU: sites annotated Conservation=Yes.'),
            ('Caveats','Neither group is the exact Muller library. The Conservation flag differs from their species rule; extracted hg19 windows are contiguous, not splice-aware.'),
            ('Binding','Motifs are selection features, not measured PABPC1 affinities. Muller Figure 3F is per-base position enrichment; Figure 2C reports 5-mer enrichment across the A15 library.'),
            ('Checksums','PAS SHA-256 '+PAS_SHA256+'; hg19 MD5 '+HG19_MD5)]
        for r in notes:s.append(list(r))
        return w
    def finish(w,path):
        for ws in w:
            ws.freeze_panes='C2';ws.row_dimensions[1].height=32
            for c in ws[1]:
                c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='173650');c.alignment=Alignment(wrap_text=True,vertical='center')
            if ws.max_row>1:ws.auto_filter.ref=ws.dimensions
            for i in range(1,ws.max_column+1):
                h=str(ws.cell(1,i).value or '');width=70 if h in ('UTR50','UTR50_A15') else 36 if h in ('utr15_A15','normalized_window_minus15_minus2') else 23
                ws.column_dimensions[get_column_letter(i)].width=width
        w.save(path)
    sample=make_summary('A/AU modification pool',[
        ('Requested positions','-13..-9; -11..-7; -6..-2'),
        ('Complete positions',len(groups['complete'])),('Two per motif positions',len(groups['balanced'])),
        ('Distinct eligible RNAs',len(groups['pool'])),('Top-two category rows',len(groups['top'])),
        ('Both-ends sheets','Omitted because the sample output is restricted to these three groups.')])
    add(sample,'Position top two',groups['top'],FIELDS)
    add(sample,'Balanced top two',[r for r in groups['top'] if r['norm_start'] in groups['balanced']],FIELDS)
    add(sample,'Grouped candidates',groups['candidates'],FIELDS)
    add(sample,'Position coverage',groups['coverage'],['norm_start','norm_end',*AU_MOTIFS,'all_five_motifs','two_per_motif'])
    add(sample,'Position counts',groups['counts'],['norm_start','norm_end','motif','eligible_unique_RNAs','selected_RNAs'])
    add(sample,'All retained AU RNAs',groups['pool'],FIELDS)
    add(sample,'All AU motif hits',groups['raw_hits'],FIELDS)
    add(sample,'Extraction failures',data['failures'],['PAS_ID','reason'])
    finish(sample,sample_path)
    counts=[]
    for group,motifs,eligible,selected in [('C/G',CG_MOTIFS,bench['cg_unique'],bench['cg']),('A/AU',AU_MOTIFS,bench['au_unique'],bench['au'])]:
        for m in motifs:counts.append({'group':group,'motif':m,'eligible_unique_RNAs':sum(r['motif']==m for r in eligible),'selected_unique_RNAs':sum(r['motif']==m for r in selected)})
    ref=bench['reference']
    b=make_summary('Matched-position benchmark',[
        ('C/G selected',len(bench['cg'])),('C/G requested',20),('A/AU selected',len(bench['au'])),('A/AU requested',20),
        ('Reference',ref['RNA_ID']+'; '+ref['Genes']+'; '+ref['PAS_ID']),('Total',len(bench['panel'])),
        ('Reference interpretation','Sequence-selected reference; no PABPC1 binding affinity was measured by this screen.'),
        ('Selection','Up to four RNAs per motif; fill remaining places using score and PAS ID. Benchmark motifs at -9 to -5.')])
    add(b,'Selected panel',bench['panel'],FIELDS)
    add(b,'Motif counts',counts,['group','motif','eligible_unique_RNAs','selected_unique_RNAs'])
    add(b,'CG eligible unique',bench['cg_unique'],FIELDS)
    add(b,'AU eligible unique',bench['au_unique'],FIELDS)
    add(b,'Extraction failures',data['failures'],['PAS_ID','reason'])
    finish(b,benchmark_path)
    return {'groups':groups,'benchmark':bench}


## 5. Include both-ends candidates and the synthetic all-A control

In [ ]:
"""Add explicit both-ends candidate sheets to the normalized sample workbook."""


def find_both(rows):
    matches=[]
    for r in rows:
        s=r['UTR50'];far=motif_at(s,-15);near=motif_at(s,-6);region=region14(s)
        if (region is None or set(region)-set('AU') or not far or not near
                or not far['retained_in_15'] or not near['retained_in_15']
                or far['motif'] not in AU_MOTIFS or near['motif'] not in AU_MOTIFS):
            continue
        matches.append({**r,'far_motif':far['motif'],'near_motif':near['motif'],
                        'far_norm_start':-15,'far_norm_end':-11,
                        'near_norm_start':-6,'near_norm_end':-2,
                        'far_junction_start':far['junction_start'],
                        'near_junction_start':near['junction_start'],
                        'Panel':'Both ends candidate'})
    return unique_candidates(matches,CONTROL_SEQUENCE)


def add_both_sheets(sample_path,data,fields):
    from openpyxl import load_workbook
    from openpyxl.styles import Font,PatternFill,Alignment
    from openpyxl.utils import get_column_letter
    both=find_both(data['hits'][-6])
    both_top=[{**r,'Panel':'Both ends top two','selection_order':i}
              for i,r in enumerate(both[:2],1)]
    extra=['far_motif','near_motif','far_norm_start','far_norm_end',
           'near_norm_start','near_norm_end','far_junction_start','near_junction_start']
    cols=fields+extra
    wb=load_workbook(sample_path)
    for name,rs in [('Both ends candidates',both),('Both ends top two',both_top)]:
        if name in wb.sheetnames:del wb[name]
        ws=wb.create_sheet(name);ws.append(cols)
        for r in rs:ws.append([r.get(c,'') for c in cols])
        ws.freeze_panes='C2'
        if ws.max_row>1:ws.auto_filter.ref=ws.dimensions
        for c in ws[1]:
            c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='173650');c.alignment=Alignment(wrap_text=True,vertical='center')
        ws.row_dimensions[1].height=32
        for i,c in enumerate(cols,1):
            ws.column_dimensions[get_column_letter(i)].width=70 if c in ('UTR50','UTR50_A15') else 36 if c=='utr15_A15' else 23
    summary=wb['Summary']
    for row in range(summary.max_row,1,-1):
        if summary.cell(row,1).value in ('Both-ends sheets','Both ends candidate count','Both ends selected','Both ends rule'):
            summary.delete_rows(row)
    summary.append(['Both ends candidate count',len(both)])
    summary.append(['Both ends selected',len(both_top)])
    summary.append(['Both ends rule','Qualifying motif at -15..-11 and another at -6..-2, both fully retained in 15 genomic UTR bases; motifs may be identical or different.'])
    wb.save(sample_path)
    return both,both_top


In [ ]:
"""Add the requested both-end categories and the explicit synthetic A30 control."""
from openpyxl import load_workbook
from openpyxl.styles import Font,PatternFill,Alignment
from openpyxl.utils import get_column_letter

def finalize_sample(sample_path,data,fields):
    both,both_top=add_both_sheets(sample_path,data,fields)
    w=load_workbook(sample_path)
    s=w['Summary']
    for row in s.iter_rows(min_row=2):
        if row[0].value=='AU-only rule':
            row[1].value='Positions -15 through -2 contain only A or U. Each selected parent has C, G, or U at -1 immediately before A15.'
    s.append(['New junction rule','For every genomic parent, UTR15 has exactly 15 bases and its final base is U, C or G, directly followed by A15.'])
    s.append(['Synthetic all-A control','A30 is explicitly retained separately. It has no non-A anchor, so Muller -1 and motif positions are undefined.'])
    s.append(['15-nt interpretation','Final 15 bases of the extracted 50-nt genomic window, not a claim that the complete transcript 3′UTR is exactly 15 nt long.'])
    partial=[]
    for start in SAMPLE_STARTS:
        for motif in AU_MOTIFS:
            candidates=unique_candidates([r for r in data['hits'][start] if r['motif']==motif],CONTROL_SEQUENCE)
            partial.extend({**r,'Panel':'Diagnostic only - incomplete motif group'} for r in candidates)
    s.append(['Diagnostic candidates','Partial position candidates are informational only; an incomplete group does not satisfy the all-five rule.'])
    s.append(['Diagnostic distinct RNA count',len({r['utr15_A15'] for r in partial})])
    diag=w.create_sheet('Partial position candidates')
    diag.append(fields)
    for r in partial:diag.append([r.get(f,'') for f in fields])
    diag.freeze_panes='C2'
    if diag.max_row>1:diag.auto_filter.ref=diag.dimensions
    for c in diag[1]:
        c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='173650');c.alignment=Alignment(wrap_text=True)
    diag.row_dimensions[1].height=32
    for i in range(1,diag.max_column+1):diag.column_dimensions[get_column_letter(i)].width=32
    name='All-A control'
    if name in w.sheetnames:del w[name]
    ws=w.create_sheet(name)
    ws.append(['RNA_ID','Panel','UTR15','utr15_A15','PAS_ID','Muller_minus1','note'])
    ws.append(['CONTROL_A30','Synthetic all-A control','A'*15,'A'*30,'','undefined','15 A assigned as UTR plus 15 A as tail; no non-A anchor.'])
    ws.freeze_panes='C2';ws.auto_filter.ref=ws.dimensions
    for c in ws[1]:
        c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='173650');c.alignment=Alignment(wrap_text=True)
    ws.row_dimensions[1].height=32
    for i in range(1,ws.max_column+1):ws.column_dimensions[get_column_letter(i)].width=48 if i==7 else 35
    w.save(sample_path)
    return both,both_top


## 6. Run the screen and download the workbooks

In [ ]:
import zipfile
from google.colab import files

print('Reading polyA_DB annotations...')
sites, input_rows = read_sites(pas_path)
print('Annotated 3′ UTR sites:', len(sites))
print('Screening hg19 sequences with Muller-normalized positions...')
reference_genome = py2bit.open(str(genome_path))
try:
    data = run_screen(sites, reference_genome)
finally:
    reference_genome.close()

output_dir = Path('Muller_junction15_results_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
output_dir.mkdir()
sample_path = output_dir / 'Muller_sample_pool_junction15.xlsx'
benchmark_path = output_dir / 'Muller_benchmark_previous_rules.xlsx'
results = make_files(sites, data, sample_path, benchmark_path)
both, both_top = finalize_sample(sample_path, data, FIELDS)
groups = results['groups']
benchmark = results['benchmark']
print('\nSample pool:')
print('Distinct eligible RNAs:', len(groups['pool']))
print('Top-two category rows:', len(groups['top']))
print('Distinct RNAs in top-two rows:', len({r['utr15_A15'] for r in groups['top']}))
print('Complete-position starts:', groups['complete'])
print('Positions with two per motif:', groups['balanced'])
print('Both-ends eligible distinct RNAs:', len(both))
if len(groups['complete']) < len(SAMPLE_STARTS):
    print('SHORTFALL: incomplete groups are omitted from top-two selections; see Position counts and Partial position candidates.')
print('The All-A control is in the separate All-A control sheet.')
print('\nBenchmark:')
print('C/G selected:', len(benchmark['cg']), '/ 20')
print('A/AU selected:', len(benchmark['au']), '/ 20')
print('Separate reference:', benchmark['reference']['Genes'], benchmark['reference']['PAS_ID'])
print('Total benchmark RNAs:', len(benchmark['panel']))
print('Genome extraction failures:', len(data['failures']))
if len(benchmark['cg'])<20 or len(benchmark['au'])<20:
    print('SHORTFALL: see the Motif counts sheet.')
zip_path=output_dir/'Muller_junction15_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    archive.write(sample_path,arcname=sample_path.name)
    archive.write(benchmark_path,arcname=benchmark_path.name)
print('\nDownloading:', zip_path)
files.download(str(zip_path))
